In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
import joblib

print("🚀 Starting Model 1 (V2) Training: The Intent-Aware Spending Forecaster")

# ==========================================
# 1. GENERATE MOCK DATA (Simulating your DB)
# ==========================================
np.random.seed(42)
num_records = 5000

# Features (X)
data = {
    'monthly_income': np.random.choice([2500, 3500, 5000, 8000], num_records),
    'savings_mode': np.random.choice(['Conservative', 'Balanced', 'Aggressive'], num_records),
    'parent_category': np.random.choice(['Food', 'Transport', 'Entertainment', 'Shopping', 'Bills'], num_records),
    'month_of_year': np.random.randint(1, 13, num_records),
    'pocket_age_days': np.random.randint(1, 365, num_records), # 1 day to 1 year old
    'previous_allocated_limit': np.random.randint(50, 1000, num_records) # What the user set last month
}

df = pd.DataFrame(data)

# Target Variable (y) Generation - Teaching the model the "User Intent" trick!
def calculate_spent(row):
    # If the pocket is BRAND NEW (< 5 days old), the user hasn't had time to spend.
    # Therefore, the "actual spend" we want to predict next month should just 
    # equal what they manually allocated (their Intent).
    if row['pocket_age_days'] <= 5:
        return row['previous_allocated_limit'] * np.random.uniform(0.95, 1.05)
    
    # If it's an OLD pocket, calculate based on lifestyle (ignoring the manual allocation)
    base_spend = row['monthly_income'] * 0.15 
    if row['parent_category'] == 'Food': base_spend *= 1.5
    elif row['parent_category'] == 'Entertainment': base_spend *= 0.8
    
    if row['savings_mode'] == 'Conservative': base_spend *= 0.7
    elif row['savings_mode'] == 'Aggressive': base_spend *= 1.3
    
    noise = np.random.normal(0, base_spend * 0.1) 
    return max(10, round(base_spend + noise, 2))

df['spent_amount'] = df.apply(calculate_spent, axis=1)

print("\n📊 Sample Data (Notice the new columns!):")
print(df[['parent_category', 'pocket_age_days', 'previous_allocated_limit', 'spent_amount']].head())

# ==========================================
# 2. FEATURE ENGINEERING
# ==========================================
le_mode = LabelEncoder()
df['savings_mode_encoded'] = le_mode.fit_transform(df['savings_mode'])

le_category = LabelEncoder()
# Notice we encode the PARENT category, not the custom name
df['parent_category_encoded'] = le_category.fit_transform(df['parent_category'])

# The New Feature Array!
X = df[['monthly_income', 'savings_mode_encoded', 'parent_category_encoded', 
        'month_of_year', 'pocket_age_days', 'previous_allocated_limit']]
y = df['spent_amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# 3. TRAIN THE MODEL
# ==========================================
print("\n🧠 Training Random Forest Regressor...")
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
print(f"✅ Model Trained! Mean Absolute Error: RM {mae:.2f}")

# ==========================================
# 4. SAVE THE MODEL & ENCODERS
# ==========================================
joblib.dump(model, 'spending_forecaster_v2.pkl')
joblib.dump(le_mode, 'encoder_savings_mode.pkl')
joblib.dump(le_category, 'encoder_parent_category.pkl')

print("\n💾 Model saved successfully as 'spending_forecaster_v2.pkl'!")

🚀 Starting Model 1 (V2) Training: The Intent-Aware Spending Forecaster

📊 Sample Data (Notice the new columns!):
  parent_category  pocket_age_days  previous_allocated_limit  spent_amount
0           Bills               15                       795        813.57
1       Transport              324                       760       1658.10
2       Transport              112                       315        516.34
3        Shopping              184                       949       1126.50
4       Transport              360                        92       1080.50

🧠 Training Random Forest Regressor...
✅ Model Trained! Mean Absolute Error: RM 68.83

💾 Model saved successfully as 'spending_forecaster_v2.pkl'!
